# A tiny Transformer tells the truth about itself

Executed report notebook. The implementation is in `experiment.py`; all displayed
numbers below are loaded from the saved run artifacts, so the notebook is useful on
GitHub without rerunning a GPU job. The final cell shows the reproduction command.

In [1]:
import csv, json
from pathlib import Path

ROOT = Path.cwd()
results = json.loads((ROOT / "artifacts/results.json").read_text())
print(json.dumps({"environment": results["environment"], "model": results["model"]}, indent=2))

{
  "environment": {
    "torch": "2.5.1+cu121",
    "device": "cuda",
    "cuda_device": "NVIDIA GeForce RTX 3070 Laptop GPU",
    "seed": 1729,
    "tf32_enabled": false
  },
  "model": {
    "vocab_size": 256,
    "max_seq_len": 64,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 2,
    "mlp_ratio": 4,
    "parameter_count": 136960,
    "tokenization": "UTF-8 bytes"
  }
}


## 1. Every tensor shape

The optimizer step contains both `B=16, T=8` and `B=16, T=64` micro-batches, with
`C=64, H=4, D=16, M=256, V=256`. Every line below states the shape, dtype, and what
each dimension means. The ledger includes both forward graphs, scaled loss
contributions, accumulated norm scalars, parameters, gradients, and tensor-valued
AdamW state.

In [2]:
print((ROOT / 'artifacts/shape_ledger.txt').read_text())

accumulation.total_token_count: shape=(), dtype=int64 | dimensions: scalar
microbatch_0_short.targets: shape=(16, 8), dtype=int64 | dimensions: batch, sequence
microbatch_0_short.input_ids: shape=(16, 8), dtype=int64 | dimensions: batch, sequence
microbatch_0_short.position_indices: shape=(8,), dtype=int64 | dimensions: sequence
microbatch_0_short.token_embedding: shape=(16, 8, 64), dtype=float32 | dimensions: batch, sequence, model channel
microbatch_0_short.position_embedding: shape=(8, 64), dtype=float32 | dimensions: sequence, model channel
microbatch_0_short.embedding_sum: shape=(16, 8, 64), dtype=float32 | dimensions: batch, sequence, model channel
microbatch_0_short.block_0.ln1: shape=(16, 8, 64), dtype=float32 | dimensions: batch, sequence, model channel
microbatch_0_short.block_0.qkv: shape=(16, 8, 192), dtype=float32 | dimensions: batch, sequence, concatenated Q/K/V channel
microbatch_0_short.block_0.q: shape=(16, 4, 8, 16), dtype=float32 | dimensions: batch, attention head, 

## 2. Finite-difference gradient check

For `lm_head.weight[105, 0]`, use the independent central difference
`(L(w+epsilon)-L(w-epsilon))/(2 epsilon)` with `epsilon=1e-5` in float64.

In [3]:
g = results["gradient_check"]
for key, value in g.items(): print(f"{key}: {value}")

parameter: lm_head.weight[105, 0]
epsilon: 1e-05
loss_plus: 5.659547824005586
loss_minus: 5.659547552736764
autograd: 0.013563441054856254
finite_difference: 0.01356344112579677
absolute_error: 7.094051561462589e-11
relative_error: 5.230274158060207e-09
agreement_decimals: 10


The absolute error is **7.094e-11** (relative error
**5.230e-09**), about 10 decimal
places of agreement.

## 3. Break gradient accumulation on purpose

Correct: `(sum(short losses)+sum(long losses))/(short tokens+long tokens)`.

Broken: `0.5*mean(short losses)+0.5*mean(long losses)`.

With lengths 8 and 64, the broken version gives the short source 50% of the gradient
instead of its correct 11.11% token weight. Both runs use identical initialization
and batches; evaluation is token weighted.

In [4]:
a = results["accumulation"]
print(f"final correct loss: {a['final_correct_eval_loss']:.6f}")
print(f"final broken loss:  {a['final_broken_eval_loss']:.6f}")
print(f"absolute gap:       {a['final_absolute_gap']:.6f}")
print(f"relative gap:       {a['final_relative_gap_percent']:.3f}%")

final correct loss: 2.944072
final broken loss:  2.980442
absolute gap:       0.036370
relative gap:       1.235%


![Correct and broken accumulation curves](artifacts/accumulation_curves.svg)

## 4. Gradient norm at every step

Global L2 norm is measured after accumulation and before the optimizer update.

In [5]:
with (ROOT / "artifacts/training_log.csv").open() as f:
    for row in csv.DictReader(f):
        print(f"step {int(row['step']):02d} | loss={float(row['correct_eval_loss']):.6f} | grad_norm={float(row['correct_grad_norm']):.6f}")

step 01 | loss=5.494010 | grad_norm=0.712102
step 02 | loss=5.308729 | grad_norm=0.769130
step 03 | loss=5.114143 | grad_norm=0.780364
step 04 | loss=4.903326 | grad_norm=0.924602
step 05 | loss=4.675746 | grad_norm=1.146403
step 06 | loss=4.444602 | grad_norm=1.195063
step 07 | loss=4.228955 | grad_norm=1.299690
step 08 | loss=4.042619 | grad_norm=1.263174
step 09 | loss=3.889215 | grad_norm=1.154972
step 10 | loss=3.763333 | grad_norm=1.104376
step 11 | loss=3.658317 | grad_norm=1.033178
step 12 | loss=3.569237 | grad_norm=0.950325
step 13 | loss=3.493037 | grad_norm=0.864298
step 14 | loss=3.428388 | grad_norm=0.714468
step 15 | loss=3.374054 | grad_norm=0.620405
step 16 | loss=3.328228 | grad_norm=0.560227
step 17 | loss=3.289713 | grad_norm=0.542839
step 18 | loss=3.257364 | grad_norm=0.613083
step 19 | loss=3.231303 | grad_norm=0.384117
step 20 | loss=3.209437 | grad_norm=0.369950
step 21 | loss=3.190977 | grad_norm=0.406815
step 22 | loss=3.175878 | grad_norm=0.247905
step 23 | 

At step 38→39, loss continues its smooth
decline `2.958693→2.952354`
(0.214%), while grad norm abruptly moves
`0.309012→0.475860`
(53.994%). The relative gradient signal is about
252 times larger. This is a leading diagnostic, not a
claim that the short run later failed.

![Gradient norms](artifacts/grad_norms.svg)

## 5. MFU

The primary calculation follows Session 10: training FLOPs/token are approximately
`6N`. An explicit architecture-aware matmul count is also reported as a cross-check.
Elementwise operations and AdamW consume time but are excluded from both numerators.

In [6]:
m = results["mfu"]
for key in ("device", "median_step_seconds", "tokens_per_second",
            "standard_6n_flops_per_token", "standard_6n_achieved_tflops",
            "standard_6n_mfu_percent", "training_matmul_flops_per_token",
            "attention_aware_mfu_percent", "peak_fp32_tflops_upper_bound"):
    print(f"{key}: {m[key]}")

device: NVIDIA GeForce RTX 3070 Laptop GPU
median_step_seconds: 0.00885995001590345
tokens_per_second: 231152.54559268133
standard_6n_flops_per_token: 821760
standard_6n_achieved_tflops: 0.18995191586624183
standard_6n_mfu_percent: 0.8833329420863181
training_matmul_flops_per_token: 786432
attention_aware_mfu_percent: 0.8453578810246631
peak_fp32_tflops_upper_bound: 21.504


Measured MFU is **0.883%**, far below 40%. Tiny matrix
multiplications, many kernel launches, explicit attention, low batch/sequence sizes,
and an unfused FP32 audit implementation dominate. The 21.504 TFLOP/s denominator is
a driver-clock upper bound, so this is an honest estimate rather than fake precision.

## 6. Decimal 0.1 in three formats

`0.1 = 1.100110011... × 2^-4` in binary; the repeating fraction must be rounded.

In [7]:
for name, item in results["float_representations"].items():
    print(f"{name:10s} {item['grouped']}  {item['hex']}  {item['stored_value']!r}")

fp32       0 01111011 10011001100110011001101  0x3DCCCCCD  0.10000000149011612
bf16       0 01111011 1001101  0x3DCD  0.10009765625
fp8_e4m3   0 0011 101  0x1D  0.1015625


I would train in **BF16** for weights/activations/gradients, retaining FP32
optimizer state and sensitive reductions. BF16 preserves FP32's exponent range and
cuts storage/bandwidth in half. Raw FP8 E4M3 is too coarse here without a carefully
tested scaling recipe. This audit run itself uses FP32 for interpretability.

## Re-run the complete experiment

Run this final cell in a CUDA-enabled environment. It regenerates JSON, CSV, the full
shape printout, and both SVG plots.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "experiment.py", "--steps", "40", "--device", "auto"], check=True)